# 12a — Crop and Climate Indices

Opens Chapter 12 (Physical Risk) with three small reference-table exercises:
`chap12_cid1` (growing-degree-days arithmetic, no data file), `chap12_cid2`
(a crop water-requirement table from Paredes et al. 2025), and `chap12_cid3`
(a table of extreme-climate-index definitions from Zhang et al. 2011).

All three MATLAB scripts build a LaTeX table string via `ftosa`/`strcat2`/
`latex_tabular` and `disp()` it — reference tables meant for typesetting in
the book, not further computation. Per this series' established convention
(see `08g`, `05d`, `10d`), we render the same content as a pandas
`DataFrame` here rather than reproducing raw LaTeX markup.

**Data note:** `cid2` and `cid3` load their tables from `.mat` files that
ship as MATLAB `table` (MCOS) objects and are opaque to `scipy.io.loadmat`
— the central Chapter 12 finding documented in `chapter12_scoping.md`. Both
have a verified-readable `.xlsx` sibling (`Data/chap12_2025_Paredes.xlsx`,
`Data/chap12_2011_Zhang.xlsx`) with an accompanying `Data/chap12_*.m`
loader script that fully specifies the sheet name and any derived columns;
we follow those loader scripts exactly.


In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "../../data"
pd.set_option("display.max_colwidth", None)


## 1. Growing degree-days (`chap12_cid1`)

A textbook growing-degree-days (GDD) calculation for a 7-day window: daily
min/max temperatures, the daily average, and GDD above a $T_{base}=10^\circ C$
threshold, then a weekly sum and a naive 12-week seasonal total
($\mathrm{GDD}_{season} = 12\times\mathrm{GDD}_{week}$, i.e. treating the
week as representative of a 12-week growing season).

In [2]:
T_min = np.array([16, 18, 14, 15, 10, 12, 15], dtype=float)
T_max = np.array([28, 32, 30, 25, 24, 30, 35], dtype=float)
T_avg = (T_max + T_min) / 2
T_base = 10.0
GDD = T_avg - T_base

days = pd.DataFrame(
    {"T_min": T_min, "T_max": T_max, "T_avg": T_avg, "GDD": GDD},
    index=[f"Day {i+1}" for i in range(7)],
).T
display(days.round(1))

GDD_week = GDD.sum()
GDD_season = 12 * GDD_week
print(f"GDD_week   = {GDD_week:.1f}")
print(f"GDD_season = {GDD_season:.1f}  (= 12 x GDD_week)")


,Day 1,Day 2,Day 3,Day 4,Day 5,Day 6,Day 7
T_min,16.0,18.0,14.0,15.0,10.0,12.0,15.0
T_max,28.0,32.0,30.0,25.0,24.0,30.0,35.0
T_avg,22.0,25.0,22.0,20.0,17.0,21.0,25.0
GDD,12.0,15.0,12.0,10.0,7.0,11.0,15.0


GDD_week   = 82.0
GDD_season = 984.0  (= 12 x GDD_week)


## 2. Crop water requirements (`chap12_cid2`)

Table 5 of Paredes et al. (2025): base and upper temperature thresholds and
crop-development-stage day counts (initial, development, mid-season,
harvest, total) for a set of vegetable crops, several with more than one
row for different season lengths (e.g. garlic's "short" vs "long" season).

The loader script (`Data/chap12_2025_Paredes.m`) splits the raw `Crop`
column — formatted as `"Name (Species)"` — into separate `Name`/`Species`
columns before the exercise script sorts by `Name` and abbreviates the
`Season` labels (e.g. `"Short season"` &#8594; `"SS"`).

In [3]:
paredes_raw = pd.read_excel(f"{DATA_DIR}/chap12_2025_Paredes.xlsx", sheet_name="Table 5")

# Port of Data/chap12_2025_Paredes.m: split "Name (Species)" -> Name, Species
name_species = paredes_raw["Crop"].str.extract(r"^(?P<Name>[^(]+?)\s*\((?P<Species>[^)]+)\)$")
paredes = pd.concat([paredes_raw, name_species], axis=1)

SEASON_ABBREV = {
    "Short season": "SS", "Long season": "LS",
    "Early maturation": "EM", "Late maturation": "LM",
    "Common": "CM", "Industry": "IND", "Market": "MKT",
}
paredes["Season"] = paredes["Season"].replace(SEASON_ABBREV)

cols = ["Name", "Species", "Season", "Base", "Upper", "Initial", "Develop", "Mid", "Harvest", "Total"]
paredes_table = paredes[cols].sort_values("Name", kind="stable").reset_index(drop=True)
paredes_table["Base"] = paredes_table["Base"].round(1)
display(paredes_table)


,Name,Species,Season,Base,Upper,Initial,Develop,Mid,Harvest,Total
0,Almond,Prunus dulcis,EM,4.5,35,250,300,2260,815,3625
1,Almond,Prunus dulcis,LM,4.5,35,280,325,2730,475,3810
2,Barley,Hordeum vulgare,SS,0.0,30,290,455,345,360,1450
3,Barley,Hordeum vulgare,LS,0.0,30,300,675,690,660,2330
4,"Bean, seed",Phaseolus vulgaris,SS,10.0,32,160,260,360,180,960
5,"Bean, seed",Phaseolus vulgaris,LS,10.0,32,320,410,400,220,1350
6,Bell pepper,Capsicum annuum,CM,10.0,35,445,1180,745,45,2415
7,Broccoli,Brassica oleracea cv. italica,SS,4.5,30,195,250,210,100,755
8,Broccoli,Brassica oleracea cv. italica,LS,4.5,30,295,350,525,110,1280
9,Canola,Brassica napus,SS,2.0,30,330,310,450,595,1685


## 3. Extreme climate indices (`chap12_cid3`)

Zhang et al. (2011)'s reference table of ETCCDI extreme-climate indices
(`TXx`, `TNx`, `TN10p`, ... ), used throughout Chapter 12 wherever these
index abbreviations appear elsewhere. The MATLAB script LaTeX-escapes the
`\u2265` symbol and the bare `"C"` units string (`->` `$^{\circ}\mathrm{C}$`)
for typesetting; the `\u2265` symbol is already plain Unicode in the source
table, and here we render `"C"` as the Unicode `\u00b0C` degree symbol
instead of LaTeX markup, since the notebook displays directly.

In [4]:
zhang = pd.read_excel(f"{DATA_DIR}/chap12_2011_Zhang.xlsx", sheet_name="Table")
zhang["Units"] = zhang["Units"].replace({"C": "\u00b0C"})
display(zhang)


,ID,Name,Definition,Units
0,TXx,Warmest day,Maximum value of daily max temperature,°C
1,TNx,Warmest night,Maximum value of daily min temperature,°C
2,TXn,Coldest day,Minimum value of daily max temperature,°C
3,TNn,Coldest night,Minimum value of daily min temperature,°C
4,TN10p,Cool nights,Percentage of time when daily min temperature < 10th percentile,%
5,TX10p,Cool days,Percentage of time when daily max temperature < 10th percentile,%
6,TN90p,Warm nights,Percentage of time when daily min temperature > 90th percentile,%
7,TX90p,Warm days,Percentage of time when daily max temperature > 90th percentile,%
8,DTR,Diurnal temperature range,Monthly mean difference between daily max and min temperature,°C
9,ETR,Extreme temperature range,Difference between TXx and TNn,°C
